# Baseline models

This notebook fits and evaluates the benchmark forecasting models on the
cleaned train, validation and test splits.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current evaluation metrics are:

1. **Cumulative log-change MAE**
2. **MASE**
3. **Relative MAE versus Persistence**
4. **Persistence win rate**

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.

In [1]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.models.modern_tcn import ModernTCNBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_evaluation_table

Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

## Load the data and clean

In [6]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

train samples: 167
val samples: 20
test samples: 62
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15
input features: ['open', 'high', 'low', 'close', 'volume', 'amount']
targets: ['close']


## Persistence

In [4]:
persistence = PersistenceBaseline.from_config(config)

persistence.fit(
    train_split=train,
    val_split=val,
)

persistence_result = persistence.predict(
    split=test,
    batch_size=256,
)

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=train,
)

persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)

for metric_name in persistence_evaluator.available_metrics:
    metric_pivot = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: The variance of predictions or target is close to zero. This can cause instability in Pearson correlationcoefficient, leading to wrong results. Consider re-scaling the input if possible or computing using alarger dtype (currently using torch.float32). Setting the correlation coefficient to nan.
  warnings.warn(*args, **kwargs)


channel,close,high,low,open
horizon,,,,
1,0.000367,0.000327,0.000326,0.000357
5,0.000785,0.000777,0.000781,0.000797
15,0.001322,0.001315,0.001324,0.001327
30,0.001838,0.001834,0.001844,0.001846
60,0.002553,0.002551,0.002565,0.002564


channel,close,high,low,open
horizon,,,,
1,nan,nan,nan,nan
5,nan,nan,nan,nan
15,nan,nan,nan,nan
30,nan,nan,nan,nan
60,nan,nan,nan,nan


channel,close,high,low,open
horizon,,,,
1,0.949704,0.973252,0.955631,0.944719
5,2.041615,2.309877,2.289573,2.114562
15,3.432314,3.899559,3.873585,3.516323
30,4.775831,5.443659,5.397603,4.894732
60,6.649479,7.589768,7.531459,6.815423


channel,close,high,low,open
horizon,,,,
1,1.000000,1.000000,1.000000,1.000000
5,1.000000,1.000000,1.000000,1.000000
15,1.000000,1.000000,1.000000,1.000000
30,1.000000,1.000000,1.000000,1.000000
60,1.000000,1.000000,1.000000,1.000000


channel,close,high,low,open
horizon,,,,
1,0.500000,0.500000,0.500000,0.500000
5,0.500000,0.500000,0.500000,0.500000
15,0.500000,0.500000,0.500000,0.500000
30,0.500000,0.500000,0.500000,0.500000
60,0.500000,0.500000,0.500000,0.500000


## Mean

In [5]:
mean = MeanBaseline.from_config(config)

mean.fit(
    train_split=train,
    val_split=val,
)

mean_result = mean.predict(
    split=test,
    batch_size=256,
)

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_pivot = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

channel,close,high,low,open
horizon,,,,
1,0.001651,0.001645,0.001653,0.001650
5,0.001787,0.001782,0.001792,0.001792
15,0.002075,0.002070,0.002081,0.002076
30,0.002429,0.002425,0.002436,0.002432
60,0.003012,0.003008,0.003025,0.003019


channel,close,high,low,open
horizon,,,,
1,0.001426,-0.015478,-0.024083,-0.003416
5,0.009109,-0.002883,0.001052,0.002187
15,0.000604,-0.003715,-0.003116,0.001543
30,0.000925,-0.001908,-0.002193,0.001197
60,-0.005952,-0.006061,-0.009094,-0.004680


channel,close,high,low,open
horizon,,,,
1,4.292428,4.883826,4.843730,4.375226
5,4.646163,5.297325,5.257017,4.760015
15,5.394405,6.148177,6.100096,5.510520
30,6.318104,7.205686,7.146998,6.459124
60,7.850684,8.955754,8.892501,8.032901


channel,close,high,low,open
horizon,,,,
1,4.477812,4.815621,4.890719,4.570597
5,2.234749,2.264760,2.254394,2.237345
15,1.553153,1.561232,1.557276,1.553968
30,1.314322,1.314827,1.316885,1.311386
60,1.171088,1.171418,1.173140,1.171546


channel,close,high,low,open
horizon,,,,
1,0.136179,0.123697,0.122268,0.134203
5,0.250662,0.249023,0.248959,0.252551
15,0.334547,0.332818,0.334050,0.333822
30,0.377161,0.376403,0.377083,0.376928
60,0.408607,0.407840,0.408118,0.408301


## ARIMA

In [4]:
arima = ArimaBaseline.from_config(
    config,
    fit_mode="simple",
    optim_method="powell",
)

arima.fit(
    train_split=train,
    val_split=val,
)

arima_result = arima.predict(
    split=test,
    batch_size=32,
)

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_pivot = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

Fitting 372 ARIMA models using fit_mode='simple'...
  fitted 25/372
  fitted 50/372
  fitted 75/372
  fitted 100/372
  fitted 125/372
  fitted 150/372
  fitted 175/372
  fitted 200/372
  fitted 225/372
  fitted 250/372
  fitted 275/372
  fitted 300/372
  fitted 325/372
  fitted 350/372
Finished fitting ARIMA models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000367,0.000327,0.000327,0.000357
5,0.000785,0.000776,0.000781,0.000797
15,0.001323,0.001315,0.001325,0.001328
30,0.001840,0.001835,0.001847,0.001848
60,0.002558,0.002553,0.002577,0.002568


channel,close,high,low,open
horizon,,,,
1,0.031713,0.097152,0.092515,0.013033
5,0.015962,0.045084,0.040774,0.001215
15,-0.001287,0.029390,0.032286,0.000559
30,0.000689,0.017838,0.020862,0.002072
60,-0.000526,0.009527,0.008055,0.000107


channel,close,high,low,open
horizon,,,,
1,0.951805,0.974656,0.957723,0.945668
5,2.042834,2.309532,2.290382,2.115750
15,3.435128,3.899825,3.877391,3.518914
30,4.781093,5.445918,5.409329,4.899085
60,6.664249,7.594147,7.572843,6.828806


channel,close,high,low,open
horizon,,,,
1,1.004039,0.999840,1.000875,1.000749
5,0.998848,0.999057,0.999103,1.000065
15,1.000198,0.999931,1.000173,1.000370
30,1.000399,1.000285,1.001035,1.000371
60,1.000953,1.000660,1.003050,1.001041


channel,close,high,low,open
horizon,,,,
1,0.458436,0.477522,0.474579,0.457172
5,0.486272,0.497736,0.492730,0.482214
15,0.489649,0.497777,0.491018,0.487499
30,0.489731,0.495340,0.487061,0.491265
60,0.485103,0.497326,0.479923,0.485012


## VAR

In [6]:
var = VarBaseline.from_config(
    config,
    maxlags=15,
    ic="aic",
    trend="c",
)

var.fit(
    train_split=train,
    val_split=val,
)

var_result = var.predict(
    split=test,
    batch_size=256,
)

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_pivot = (
        var_metric_table
        .loc[
            var_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

Fitting 4 VAR model(s) with maxlags=15, ic=aic...
  open: selected_lag=12, failed=False
  high: selected_lag=11, failed=False
  low: selected_lag=10, failed=False
  close: selected_lag=11, failed=False
Finished fitting VAR models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000379,0.000336,0.000335,0.000367
5,0.000799,0.000789,0.000794,0.000812
15,0.001335,0.001326,0.001336,0.001343
30,0.001848,0.001841,0.001853,0.001857
60,0.002563,0.002557,0.002580,0.002574


channel,close,high,low,open
horizon,,,,
1,0.010872,0.072522,0.069605,0.044040
5,0.015784,0.024825,0.028767,0.018205
15,-0.002537,0.013403,0.010317,0.000302
30,0.004379,0.020538,0.014423,0.008814
60,0.002263,0.016264,0.012470,0.008423


channel,close,high,low,open
horizon,,,,
1,0.982359,0.999564,0.982341,0.972873
5,2.076806,2.347173,2.323892,2.153473
15,3.464684,3.931396,3.906929,3.557393
30,4.800755,5.464885,5.428472,4.922594
60,6.675028,7.605355,7.579781,6.843709


channel,close,high,low,open
horizon,,,,
1,1.039215,1.028317,1.026234,1.024637
5,1.019413,1.017801,1.015719,1.018847
15,1.008641,1.007648,1.008572,1.010811
30,1.004279,1.002999,1.004848,1.004872
60,1.003097,1.003059,1.004991,1.004573


channel,close,high,low,open
horizon,,,,
1,0.434530,0.455022,0.452580,0.449742
5,0.469837,0.473127,0.472968,0.469221
15,0.475889,0.478476,0.477568,0.472831
30,0.482826,0.488992,0.482707,0.483205
60,0.484953,0.491452,0.479773,0.482976


## GARCH

In [7]:
garch = GarchBaseline.from_config(
    config,
    mean="AR",
    return_scale=10000.0,
)

garch.fit(
    train_split=train,
    val_split=val,
)

garch_result = garch.predict(
    split=test,
    batch_size=256,
)

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_pivot = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"] == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

Fitting 372 GARCH(1,1) models with mean='AR'...
  fitted 25/372
  fitted 50/372
  fitted 75/372
  fitted 100/372
  fitted 125/372
  fitted 150/372
  fitted 175/372
  fitted 200/372
  fitted 225/372
  fitted 250/372
  fitted 275/372
  fitted 300/372
  fitted 325/372
  fitted 350/372
Finished fitting GARCH models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000368,0.000328,0.000327,0.000357
5,0.000785,0.000777,0.000782,0.000797
15,0.001323,0.001316,0.001324,0.001328
30,0.001841,0.001839,0.001844,0.001848
60,0.002560,0.002565,0.002566,0.002569


channel,close,high,low,open
horizon,,,,
1,0.031935,0.090563,0.089381,0.011004
5,0.014379,0.043186,0.038127,0.005380
15,0.000941,0.028866,0.033691,0.005429
30,0.002778,0.014833,0.026356,0.006980
60,0.003407,0.004318,0.020248,0.007641


channel,close,high,low,open
horizon,,,,
1,0.952390,0.974924,0.957227,0.945705
5,2.042920,2.312340,2.290399,2.116174
15,3.435790,3.905109,3.874415,3.518939
30,4.782485,5.458148,5.399424,4.899692
60,6.668193,7.632369,7.537462,6.831841


channel,close,high,low,open
horizon,,,,
1,1.005828,1.001511,1.001202,1.000983
5,0.999117,1.000034,0.999613,1.000280
15,1.000455,1.001517,0.999991,1.000556
30,1.000854,1.002106,1.000278,1.000585
60,1.001783,1.004740,1.000684,1.001701


channel,close,high,low,open
horizon,,,,
1,0.456903,0.467527,0.470471,0.454164
5,0.485596,0.483072,0.490041,0.476811
15,0.487458,0.483912,0.495080,0.487025
30,0.487705,0.482141,0.495943,0.489179
60,0.482064,0.471557,0.496878,0.484314


In [9]:
modern_tcn_checkpoint_path = Path(
    '/Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/checkpoints/modern_tcn/joint_c/runs/88l8gd68/best_checkpoint.pt'
).expanduser().resolve()

modern_tcn = ModernTCNBaseline.from_config(config)

modern_tcn.load_checkpoint(
    checkpoint_path=modern_tcn_checkpoint_path,
    device="cpu",
)

modern_tcn_result = modern_tcn.predict(
    split=test,
    batch_size=8,
    num_workers=0,
)

modern_tcn_evaluator = ForecastEvaluator(
    prediction_result=modern_tcn_result,
    train_split=train,
)

modern_tcn_results = modern_tcn_evaluator.evaluate(
    metrics=modern_tcn_evaluator.available_metrics,
    reduce_dims=(0, 2),
)

modern_tcn_metric_table = make_evaluation_table(
    metric_results=modern_tcn_results,
    horizons=modern_tcn_evaluator.horizons,
    channels=modern_tcn_evaluator.channels,
)

for metric_name in modern_tcn_evaluator.available_metrics:
    metric_pivot = (
        modern_tcn_metric_table
        .loc[
            modern_tcn_metric_table["metric"]
            == metric_name
        ]
        .pivot(
            index="horizon",
            columns="channel",
            values="value",
        )
    )

    display(
        metric_pivot.style.set_caption(metric_name)
    )

channel,close
horizon,
1,0.000371
5,0.000788
15,0.001326
30,0.001842
60,0.002562


channel,close
horizon,
1,0.019874
5,0.015445
15,0.021134
30,0.019636
60,0.011261


channel,close
horizon,
1,0.962564
5,2.048526
15,3.441900
30,4.787201
60,6.674634


channel,close
horizon,
1,1.013245
5,1.002554
15,1.002454
30,1.001871
60,1.003845


channel,close
horizon,
1,0.449605
5,0.483802
15,0.484309
30,0.487139
60,0.483935
